In [1]:
import torch
import torch.nn as nn
import torchinfo
from model.mini_cnn import MiniCNN
from result.repondeur import prediction_to_csv
import torchvision
import numpy as np
import csv
from tqdm import tqdm
from model.loader import load_dataset, load_test_loader

In [2]:
# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU - training will be slow!")

⚠ Using CPU - training will be slow!


In [3]:
def train_crossEntropyLoss(train_set, model, epochs):
    """
    Entraine un model à partir d'un train_set donné. Retourne les metrics de l'entrainement
    """

    # Parameters
    model.train()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.1,
        momentum=0.9,
        weight_decay=5e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=200
    )


    accuracies = np.zeros(epochs)
    losses = np.zeros(epochs)

    for epoch in range(epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(
            train_set,
            desc=f"Model training | Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            #prediction
            logits = model(images)
            loss = criterion(logits, labels)

            #propagation du gradient
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Calcul des metriques
            total_loss += loss.item()
            
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({
                "batch_loss": f"{loss.item():.4f}",
                "accuracy": f"{correct/total*100:.2f}%"
            })
        accuracies[epoch] = correct/total
        losses[epoch] = total_loss

        scheduler.step()
    return accuracies, losses

In [13]:
from torch.utils.data import DataLoader
from torchvision import transforms
from utils.CustomDataset import CustomDataset
from torchvision.transforms import v2

In [17]:
# 1. Define Image Transforms
# HUUUUGOOOOOOOO
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     torchvision.transforms.functional.to_dtype(float16, scale=True),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

transform = v2.Compose([
    #v2.ToImage(),                  # Convert to Tensor (if it's PIL)
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True), # Scales to float16 [0, 1] # TODO
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# 2. Instantiate your Dataset
train_dataset = CustomDataset(
    img_dir='data/',
    annotations_file='data/train.csv',
    transform=transform, 
    target_transform=None # ???
)

In [18]:

epochs = 5
model = MiniCNN()
model.to(device)
train_set = DataLoader(
    dataset=train_dataset,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

training_curves, training_curves = train_crossEntropyLoss(train_set,model, epochs)


Model training | Epoch 1/5:   0%|          | 0/250 [00:00<?, ?it/s]/home/niels/.sdd_venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Model training | Epoch 1/5: 100%|██████████| 250/250 [02:42<00:00,  1.54it/s, batch_loss=2.7211, accuracy=14.96%]
Model training | Epoch 2/5: 100%|██████████| 250/250 [02:40<00:00,  1.56it/s, batch_loss=2.4051, accuracy=21.80%]
Model training | Epoch 3/5: 100%|██████████| 250/250 [02:46<00:00,  1.50it/s, batch_loss=2.3947, accuracy=25.33%]
Model training | Epoch 4/5: 100%|██████████| 250/250 [02:39<00:00,  1.56it/s, batch_loss=2.5304, accuracy=28.76%]
Model training | Epoch 5/5: 100%|██████████| 250/250 [02:48<00:00,  1.48it/s, batch_loss=2.5720, accuracy=30.92%]


In [20]:
torch.save(model.state_dict(), f"model/trained_model/MiniCNN_{epochs}_epochs.pth")